# Bài học 08 - Mẫu Thiết kế Đa Tác nhân


## Cài đặt


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tại sao lại là Hệ thống Đa Tác nhân?

Các nhiệm vụ thực tế như lập kế hoạch chuyến đi đòi hỏi nhiều loại chuyên môn khác nhau — logistics, kiến thức địa phương, lập ngân sách và nhiều hơn thế nữa. Một tác nhân đơn lẻ cố gắng xử lý tất cả mọi thứ sẽ nhanh chóng trở nên không hiệu quả.

Hệ thống đa tác nhân giải quyết vấn đề này thông qua **sự chuyên môn hóa**: mỗi tác nhân tập trung vào một lĩnh vực chuyên môn, tạo ra kết quả chất lượng cao hơn so với một người làm đa năng. Chúng cũng cải thiện **khả năng mở rộng** — bạn có thể thêm các tác nhân mới (ví dụ, chuyên gia bay, nhà phê bình nhà hàng) mà không cần viết lại quy trình làm việc hiện có. Các tác nhân kết hợp với nhau thông qua một chuỗi có cấu trúc, truyền ngữ cảnh từ tác nhân này sang tác nhân khác.


## Tạo ra các Đại lý Chuyên biệt


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Xây dựng một luồng công việc tuần tự

`WorkflowBuilder` cho phép bạn kết nối các đại lý thành một đồ thị có hướng. Ở đây chúng ta tạo một pipeline hai bước đơn giản: **TravelPlanner** soạn thảo hành trình, sau đó **TravelConcierge** xem xét và cải thiện nó.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Thêm nhiều tác nhân hơn vào quy trình làm việc

Một trong những lợi thế lớn nhất của mẫu đa tác nhân là sự dễ dàng mở rộng. Dưới đây chúng tôi thêm một tác nhân **BudgetReviewer** kiểm tra kế hoạch so với ngân sách của người đi du lịch, đánh dấu các mục có thể làm vượt quá chi phí và đề xuất các lựa chọn thay thế tiết kiệm tiền. Quy trình làm việc bây giờ chạy ba tác nhân theo trình tự:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Tóm tắt

Trong bài học này, bạn đã học cách:

1. **Tạo các tác nhân chuyên biệt** — mỗi tác nhân đảm nhận một vai trò cụ thể (lập kế hoạch, hỗ trợ khách hàng, xem xét ngân sách).
2. **Kết nối các tác nhân thành một quy trình làm việc tuần tự** bằng cách sử dụng `WorkflowBuilder` và `add_edge`.
3. **Phát trực tiếp đầu ra** từ một chuỗi đa tác nhân, theo dõi tác nhân nào đang nói.
4. **Mở rộng một quy trình làm việc** bằng cách thêm các tác nhân mới vào chuỗi mà không sửa đổi các tác nhân hiện có.

Mẫu thiết kế đa tác nhân giữ cho mỗi tác nhân đơn giản trong khi tạo ra các kết quả phong phú và được xem xét kỹ lưỡng hơn so với bất kỳ tác nhân đơn lẻ nào có thể đạt được một mình.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:  
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi nỗ lực đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc sự không chính xác. Tài liệu gốc bằng ngôn ngữ bản địa nên được xem là nguồn đáng tin cậy và chính thức. Đối với các thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu lầm hay diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
